# LELA60341: Research Methods in Computational and Corpus Linguistics Week 1

The content of this workbook is based on this book:
https://www.librarysearch.manchester.ac.uk/permalink/44MAN_INST/1l23p04/alma992992666536101631

In [11]:
# ▶ Run this cell first. It defines the check() helper used by every exercise.
def check(label, condition):
    """Print a tick or a cross instead of crashing, so you can keep working."""
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as err:
        passed = False
        label += f"  (error: {type(err).__name__}: {err})"
    print(("✅ " if passed else "❌ ") + label)
    return passed

---
# Python overview

## The Python interpreter

Python is an **interpreted** language. A program called the **interpreter** reads each command, carries it out, and reports the result. There are two common ways to use it:

* **Interactively**: type a command and see the answer at once. This notebook is an interactive environment.
* **Scripts**: save commands in a text file ending in `.py` (for example `demo.py`) and run it with `python demo.py`. Running `python -i demo.py` runs the script and then leaves you in interactive mode, with all its variables still available.

IDEs such as IDLE (which ships with Python), VS Code, PyCharm or Positron add an editor and a debugger on top of the interpreter.


In [7]:
typed = ["A", "B+", "Z", "A-", ""]      # pretend keyboard entries; "" is the blank line that ends input
def fake_input():
    return typed.pop(0)                  # hand back the next pretend entry

print("Welcome to the GPA calculator.")
print("Please enter all your letter grades, one per line.")
print("Enter a blank line to designate the end.")
# map from letter grade to point value
points = {'A+': 4.0, 'A': 4.0, 'A-': 3.67, 'B+': 3.33, 'B': 3.0, 'B-': 2.67,
          'C+': 2.33, 'C': 2.0, 'C-': 1.67, 'D+': 1.33, 'D': 1.0, 'F': 0.0}
num_courses = 0
total_points = 0
done = False
while not done:
    grade = fake_input()                 # read line from "user"
    if grade == '':                      # empty line was entered
        done = True
    elif grade not in points:            # unrecognized grade entered
        print("Unknown grade {0} being ignored".format(grade))
    else:
        num_courses += 1
        total_points += points[grade]
if num_courses > 0:                      # avoid division by zero
    print("Your GPA is {0:.3}".format(total_points / num_courses))

Welcome to the GPA calculator.
Please enter all your letter grades, one per line.
Enter a blank line to designate the end.
Unknown grade Z being ignored
Your GPA is 3.67


Everything in this program will become clear over the coming weeks. For now, notice three things about how Python code **looks**:

* **Whitespace matters.** Each statement normally ends at the end of its line. A statement can carry on to the next line if a bracket is still open (like the `{` of `points` above) or if the line ends with a backslash `\`.
* **Indentation marks blocks.** The eight lines indented under `while not done:` are the body of the loop. The `if/elif/else` inside is indented further, because it is *nested* inside the loop.
* **Comments** start with `#`. Python ignores everything from `#` to the end of the line.


---
# Objects in Python

A piece of data held in memory in Python is referred to as an object. Each object has a class (sometimes called a type).

## Identifiers, objects and the assignment statement

The most common statement in Python is an **assignment**:

```python
temperature = 98.6
```

Read it from **right to left**:

1. Python evaluates the right-hand side. The characters `98.6` are a *floating-point literal*, so Python creates a `float` object with that value.
2. The name on the left, `temperature`, is an **identifier** (or *name*). It is bound to that object.

```
temperature ───────►  ┌─────────┐
                      │  float  │
                      │  98.6   │
                      └─────────┘
```

The name is not the object. It is a label that **refers to** the object. Each name is associated with the memory address of its object, and `id()` lets us look at that address.

In [8]:
temperature = 98.6

print(temperature)        # the value of the object
print(type(temperature))  # the class of the object
print(id(temperature))    # the object's identity (its address in CPython)

98.6
<class 'float'>
136381840773552


### Rules for identifiers

* Identifiers are **case-sensitive**: `temperature` and `Temperature` are two different names.
* They can mix letters, digits and underscores (and other Unicode letters), but **cannot start with a digit**, so `9lives` is illegal.
* **Reserved words** (keywords) can't be used as names.

In [9]:
import keyword

print(len(keyword.kwlist), "reserved words in this version of Python:")
print(keyword.kwlist)

35 reserved words in this version of Python:
['False', 'None', 'True', 'and', 'as', 'assert', 'async', 'await', 'break', 'class', 'continue', 'def', 'del', 'elif', 'else', 'except', 'finally', 'for', 'from', 'global', 'if', 'import', 'in', 'is', 'lambda', 'nonlocal', 'not', 'or', 'pass', 'raise', 'return', 'try', 'while', 'with', 'yield']


### Dynamic typing

Python is **dynamically typed**. You never declare that a name "is a float". A name can refer to *any* object and can later be re-bound to an object of a completely different type. **The name has no type. The object does.**

In [10]:
x = 98.6
print(type(x))
x = "now I'm a string"
print(type(x))
x = None          # None is a special object meaning "nothing here"
print(type(x))

<class 'float'>
<class 'str'>
<class 'NoneType'>


### Aliases

Assigning one name to another does **not** copy the object. It creates an **alias**, a second name for the *same* object:

```python
original = temperature
```

```
temperature ──►  ┌───────┐  ◄── original
                 │ 98.6  │
                 └───────┘
```

The `is` operator asks: *do these two names refer to the very same object?*

In [14]:
temperature = 98.6
original = temperature

print(original is temperature)          # same object?
print(id(original) == id(temperature))  # the same test, using id()

True
True


### Reassignment breaks an alias. It does not change the object.

🔮 **Predict:** after running `temperature = temperature + 5.0`, what is `original`?

In [15]:
temperature = 98.6
original = temperature

temperature = temperature + 5.0   # RHS is evaluated first: 98.6 + 5.0 -> a NEW float object

print("temperature:", temperature)
print("original:   ", original)
print("still aliases?", temperature is original)

temperature: 103.6
original:    98.6
still aliases? False


Here is what happened, step by step:

1. `temperature + 5.0` was evaluated using the *existing* binding, which produced a **new** `float` with value 103.6.
2. Only then was the name `temperature` moved onto that new object.
3. The old `98.6` object was never touched, and `original` still points at it.

```
┌───────┐                          ┌───────┐
│ 103.6 │ ◄── temperature          │ 98.6  │ ◄── original
└───────┘                          └───────┘
```

> 💡 **Key idea:** *assignment moves names. It never changes objects.* Whether an object can be changed through one of its names depends on its class (mutable or immutable), which is the topic of Section 1.2.3.

### 🏋️ Exercise 1 · Practice — Trace the names

Read the code **without running it** and fill in the values you expect for `a`, `b` and `c`, and whether `a is b`. Then run the cell to check your trace.

In [22]:
a = 10
b = a
a = a + 1
c = b
b = b * 2

# Your predictions (replace None):
predict_a = 11
predict_b = 20
predict_c = 10
predict_a_is_b = False   # True or False

In [23]:
# ✅ Check
check("a", lambda: predict_a == a)
check("b", lambda: predict_b == b)
check("c", lambda: predict_c == c)
check("a is b", lambda: predict_a_is_b == (a is b))

✅ a
✅ b
✅ c
✅ a is b


True

---
## Creating and using objects

### Instantiation

Creating a new object of a class is called **instantiation**. There are three ways to do it:

| Way | Example | Notes |
|---|---|---|
| Call the class's **constructor** | `list()`, `int("42")`, `Widget(a, b, c)` | Put `()` after the class name, plus any parameters |
| Write a **literal** | `98.6`, `"hello"`, `[1, 2]` | Built-in classes have a special shorthand syntax |
| Call a **function that returns** a new instance | `sorted(data)` | The function builds the object and hands it back |

In [ ]:
a = float()               # constructor with no parameters -> 0.0
b = float("2.5")          # constructor with a parameter
c = 2.5                   # literal form
d = sorted([3, 1, 2])     # a function that returns a brand-new list

print(a, b, c, d)
print(type(d))

## Functions and methods

A function in Python is a named, reusable block of code, that takes inputs called parameters, does some work, and can send back a result with return (or None if it doesn't). Functions are objects too, so they can be given other names, passed to other functions (as in max(values, key=abs)) and stored in lists or dictionaries like any other value.

A **function** is called on its own: `sorted(data)`.
A **method** (or *member function*) belongs to a class and is called *on an object* with the dot operator: `data.sort()`.

The thing to the left of the dot doesn't have to be a name. It can be any expression that produces an object, so calls can be **chained**:


In [ ]:
response = "  Yes please  "
print(response.strip().lower().startswith("y"))
#        └─ new str ─┘
#               └─ new str ─┘
#                        └── called on the result of .lower()

### Accessors vs mutators

It matters a lot whether a method *changes* the object it is called on:

* **Accessors** report information and leave the object alone.
* **Mutators** (or *update methods*) change the object's state.

The classic pair is `sorted(data)` (a function that returns a *new* list) and `data.sort()` (a mutator that rearranges `data` *in place* and returns `None`).

🔮 **Predict** the four printed lines before running.

In [ ]:
data = [5, 2, 9, 1]

result = sorted(data)      # function: builds a new list
print("sorted() returned:", result)
print("data afterwards:  ", data)

result = data.sort()       # mutator: changes data itself
print("sort() returned:  ", result)
print("data afterwards:  ", data)

> ⚠️ **Common bug:** `data = data.sort()` replaces your list with `None`, because mutators usually return `None`.

### 🏋️ Exercise 2 · Practice — The `sort()` trap

This function is supposed to return a sorted copy of `scores` **without changing the caller's list**. It has **two** bugs. Fix it.

In [ ]:
def top_three(scores):
    """Return the three highest scores, highest first. Must not modify `scores`."""
    scores = scores.sort(reverse=True)
    return scores[:3]

my_scores = [55, 92, 71, 88, 64]
print(top_three(my_scores))
print(my_scores)

[92, 88, 71]
[92, 88, 71, 64, 55]


In [1]:
# ✅ Check
data = [55, 92, 71, 88, 64]
check("returns correct top three", lambda: top_three(list(data)) == [92, 88, 71])
def _leaves_original_alone():
    original = [5, 3, 9, 1]
    top_three(original)
    return original == [5, 3, 9, 1]
check("caller's list is unchanged", _leaves_original_alone)

NameError: name 'check' is not defined

## Modules and Packages


A module is a file of Python code, that groups related definitions such as functions, classes and variables so they can be reused in other programs. You can create these yourself or use others'. When you import it, Python runs that file once and creates a module object whose attributes are the names the file defined, so you use them with dot notation, for example math.sqrt(2). Each module has its own namespace, so its names don't clash with yours unless you import them directly with from math import sqrt.

A package is a folder of related modules grouped under one name. Python's standard library is a large collection of modules and packages (math, random, re, os and so on) that come with every installation.
